# Project Sentinel — Phase 2: Data Understanding

Live pull from the NASA NeoWs feed endpoint (two chained 7-day windows), followed by native-Python (no pandas/numpy) structural and quality audits, per the project brief.



In [6]:
import sys
sys.path.insert(0, '../src')
sys.path.insert(0, '..')

import json
from pipeline import (
    fetch_neos, extract_and_write_ids, scrape_total_known_neos,
    walk_leaf_types, numeric_field_min_max_mean, quality_verification,
    DEFAULT_DATE_WINDOWS, API_KEY, safe_float,
)


## 1. Pull the live NEO catalog

Two chained 7-day windows (the API's hard per-call limit), merged into one running list. `near_earth_objects` is a dict keyed by date string, not a flat list — `fetch_neos` loops over `.items()` before reaching an actual object.

In [7]:
records = fetch_neos(DEFAULT_DATE_WINDOWS, API_KEY)
len(records)


[fetch_neos] 2026-08-01..2026-08-07: 29 objects.
[fetch_neos] 2026-08-08..2026-08-14: 37 objects.
[fetch_neos] Total merged objects across all windows: 66.


66

In [8]:
neo_ids = extract_and_write_ids(records)
neo_ids[:5]


[extract_and_write_ids] Extracted 66 unique NEO ids -> D:\DEPI__5\project_Sentinel\project_repo_sentinel\data\raw\extracted_ids.txt


['2136770', '3754379', '3991578', '3653190', '3709137']

## 2. Structural audit — recursive leaf-type walk

Run on several different records, not just one — this is where the string-vs-number inconsistency inside `close_approach_data` (real numbers in `estimated_diameter`, but quoted strings for `relative_velocity` / `miss_distance`) becomes visible.

In [9]:
for i, record in enumerate(records[:5]):
    print(f'--- record {i} ---')
    walk_leaf_types(record)
    print()


--- record 0 ---
root.links.self: str = 'http://api.nasa.gov/neo/rest/v1/neo/2136770?api_key=DEMO_KEY'
root.id: str = '2136770'
root.neo_reference_id: str = '2136770'
root.name: str = '136770 (1996 PC1)'
root.nasa_jpl_url: str = 'https://ssd.jpl.nasa.gov/tools/sbdb_lookup.html#/?sstr=2136770'
root.absolute_magnitude_h: float = 20.56
root.estimated_diameter.kilometers.estimated_diameter_min: float = 0.2053784995
root.estimated_diameter.kilometers.estimated_diameter_max: float = 0.459240286
root.estimated_diameter.meters.estimated_diameter_min: float = 205.3784995184
root.estimated_diameter.meters.estimated_diameter_max: float = 459.2402860401
root.estimated_diameter.miles.estimated_diameter_min: float = 0.1276162436
root.estimated_diameter.miles.estimated_diameter_max: float = 0.2853585958
root.estimated_diameter.feet.estimated_diameter_min: float = 673.8139963601
root.estimated_diameter.feet.estimated_diameter_max: float = 1506.6939000519
root.is_potentially_hazardous_asteroid: bool = 

In [10]:
print(json.dumps(records[0], indent=2) if records else 'no records')

{
  "links": {
    "self": "http://api.nasa.gov/neo/rest/v1/neo/2136770?api_key=DEMO_KEY"
  },
  "id": "2136770",
  "neo_reference_id": "2136770",
  "name": "136770 (1996 PC1)",
  "nasa_jpl_url": "https://ssd.jpl.nasa.gov/tools/sbdb_lookup.html#/?sstr=2136770",
  "absolute_magnitude_h": 20.56,
  "estimated_diameter": {
    "kilometers": {
      "estimated_diameter_min": 0.2053784995,
      "estimated_diameter_max": 0.459240286
    },
    "meters": {
      "estimated_diameter_min": 205.3784995184,
      "estimated_diameter_max": 459.2402860401
    },
    "miles": {
      "estimated_diameter_min": 0.1276162436,
      "estimated_diameter_max": 0.2853585958
    },
    "feet": {
      "estimated_diameter_min": 673.8139963601,
      "estimated_diameter_max": 1506.6939000519
    }
  },
  "is_potentially_hazardous_asteroid": false,
  "close_approach_data": [
    {
      "close_approach_date": "2026-08-06",
      "close_approach_date_full": "2026-Aug-06 05:03",
      "epoch_date_close_approach"

## 3. Custom EDA — native single-pass min/max/mean

One numeric field at a time, running accumulators only (no `min()`/`max()`/`sum()`). `relative_velocity` and `miss_distance` both need a `safe_float()` cast first since NASA returns them as strings.

In [11]:
max_diam_extractor = lambda r: r.get('estimated_diameter', {}).get('kilometers', {}).get('estimated_diameter_max')
mn, mx, mean, n_missing = numeric_field_min_max_mean(records, max_diam_extractor)
print(f'max_diameter_km: min={mn}  max={mx}  mean={mean:.4f}  missing={n_missing}' if mean is not None else 'no data')


max_diameter_km: min=0.008590926  max=5.2485577338  mean=0.3204  missing=0


In [12]:
def first_approach(r):
    approaches = r.get('close_approach_data') or []
    return approaches[0] if approaches else None

miss_dist_extractor = lambda r: (first_approach(r) or {}).get('miss_distance', {}).get('kilometers')
mn, mx, mean, n_missing = numeric_field_min_max_mean(records, miss_dist_extractor)
print(f'miss_distance_km: min={mn}  max={mx}  mean={mean:.1f}  missing={n_missing}' if mean is not None else 'no data')


miss_distance_km: min=2515647.475861976  max=73998365.3347275  mean=42493329.5  missing=0


In [13]:
velocity_extractor = lambda r: (first_approach(r) or {}).get('relative_velocity', {}).get('kilometers_per_hour')
mn, mx, mean, n_missing = numeric_field_min_max_mean(records, velocity_extractor)
print(f'relative_velocity_kph: min={mn}  max={mx}  mean={mean:.1f}  missing={n_missing}' if mean is not None else 'no data')


relative_velocity_kph: min=6483.3012142977  max=130940.1471982291  mean=48942.1  missing=0


## 4. Quality verification

% of records missing/empty `close_approach_data` and `absolute_magnitude_h`, plus confirming `is_potentially_hazardous_asteroid` is present and boolean-typed on every record before the Validation Check relies on it.

In [14]:
quality = quality_verification(records)
print(json.dumps(quality, indent=2))


{
  "pct_missing_close_approach_data": 0.0,
  "pct_missing_absolute_magnitude_h": 0.0,
  "n_hazard_flag_present_and_bool": 66,
  "n_total": 66
}


## 5. Bonus scrape — total known NEOs

Finds the anchor phrase `"Total number of discovered near-Earth asteroids"` on NASA's Planetary Defense page and reads the number sitting just *before* it (immediately in front of a colon). This value is attached as a constant `total_known_neos` column on every row in Phase 3.

In [15]:
total_known_neos = scrape_total_known_neos()
total_known_neos


[scrape_total_known_neos] Window before anchor: '2025.   Key statistics include: 39,123: '
[scrape_total_known_neos] Parsed total_known_neos = 39123


39123

## Notes for README

- Record the min/max/mean figures and quality-verification percentages above into the README's Phase 1/2 sections once this notebook has been run against the live API.
- `total_known_neos` will differ from anyone else's — NASA updates that page monthly. That's expected, not a bug; use whatever value your own run scrapes for the README's summary insight.